# Feature engineering - advanced data preparation pipeline  | Sebislaw

## Libraries

In [1]:
from os.path  import join
import random
import itertools
import math

import numpy as np
import pandas as pd
from pandas.plotting import scatter_matrix

import matplotlib.pyplot as plt
import plotly.express as px
from pandas.plotting import parallel_coordinates
import seaborn as sns
import ipywidgets as widgets
from IPython.display import display

from sklearn.linear_model import LinearRegression, LassoCV, LogisticRegression, LogisticRegressionCV
from sklearn.model_selection import train_test_split, TimeSeriesSplit, cross_val_score, StratifiedKFold, RandomizedSearchCV
import statsmodels.api as sm
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import brier_score_loss
from sklearn.feature_selection import SelectFromModel
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.calibration import CalibratedClassifierCV
from sklearn.pipeline import Pipeline
from sklearn.feature_selection import mutual_info_classif
from sklearn.neural_network import MLPClassifier

import xgboost as xgb
from xgboost import XGBClassifier
from catboost import CatBoostClassifier
import optuna
# from tabpfn import TabPFNClassifier

## Data

In [2]:
data_path = '..\\..\\..\\data'
pd.set_option('display.max_columns', None)

# The Basics ------------------------------------------------------------------------
# Men
MTeams = pd.read_csv(join(data_path, 'MTeams.csv'))
MSeasons = pd.read_csv(join(data_path, 'MSeasons.csv'))
MNCAATourneySeeds = pd.read_csv(join(data_path, 'MNCAATourneySeeds.csv'))
MRegularSeasonCompactResults = pd.read_csv(join(data_path, 'MRegularSeasonCompactResults.csv'))
MNCAATourneyCompactResults = pd.read_csv(join(data_path, 'MNCAATourneyCompactResults.csv'))
# Women
WTeams = pd.read_csv(join(data_path, 'WTeams.csv'))
WSeasons = pd.read_csv(join(data_path, 'WSeasons.csv'))
WNCAATourneySeeds = pd.read_csv(join(data_path, 'WNCAATourneySeeds.csv'))
WRegularSeasonCompactResults = pd.read_csv(join(data_path, 'WRegularSeasonCompactResults.csv'))
WNCAATourneyCompactResults = pd.read_csv(join(data_path, 'WNCAATourneyCompactResults.csv'))
# Other
SampleSubmissionStage1 = pd.read_csv(join(data_path, 'SampleSubmissionStage1.csv'))
SampleSubmissionStage2 = pd.read_csv(join(data_path, 'SampleSubmissionStage2.csv'))
SeedBenchmarkStage1 = pd.read_csv(join(data_path, 'SeedBenchmarkStage1.csv'))

# Team Box Scores ------------------------------------------------------------------------
# Men
MRegularSeasonDetailedResults = pd.read_csv(join(data_path, 'MRegularSeasonDetailedResults.csv'))
MNCAATourneyDetailedResults = pd.read_csv(join(data_path, 'MNCAATourneyDetailedResults.csv'))
# Women
WRegularSeasonDetailedResults = pd.read_csv(join(data_path, 'WRegularSeasonDetailedResults.csv'))
WNCAATourneyDetailedResults = pd.read_csv(join(data_path, 'WNCAATourneyDetailedResults.csv'))

# Geography ------------------------------------------------------------------------
# All
Cities = pd.read_csv(join(data_path, 'Cities.csv'))
Conferences = pd.read_csv(join(data_path, 'Conferences.csv'))
# Men
MGameCities = pd.read_csv(join(data_path, 'MGameCities.csv'))
# Women
WGameCities = pd.read_csv(join(data_path, 'WGameCities.csv'))

# Public Rankings ------------------------------------------------------------------------
# Men
MMasseyOrdinals = pd.read_csv(join(data_path, 'MMasseyOrdinals.csv')) # men only

# Supplements ------------------------------------------------------------------------
# Men
MTeamCoaches = pd.read_csv(join(data_path, 'MTeamCoaches.csv')) # men only
MTeamConferences = pd.read_csv(join(data_path, 'MTeamConferences.csv'))
MConferenceTourneyGames = pd.read_csv(join(data_path, 'MConferenceTourneyGames.csv'))
MSecondaryTourneyTeams = pd.read_csv(join(data_path, 'MSecondaryTourneyTeams.csv'))
MSecondaryTourneyCompactResults = pd.read_csv(join(data_path, 'MSecondaryTourneyCompactResults.csv'))
MTeamSpellings = pd.read_csv(join(data_path, "MTeamSpellings.csv"), encoding='cp1252')
MNCAATourneySlots = pd.read_csv(join(data_path, 'MNCAATourneySlots.csv'))
MNCAATourneySeedRoundSlots = pd.read_csv(join(data_path, 'MNCAATourneySeedRoundSlots.csv')) # men only
# Women
WTeamConferences = pd.read_csv(join(data_path, 'WTeamConferences.csv'))
WConferenceTourneyGames = pd.read_csv(join(data_path, 'WConferenceTourneyGames.csv'))
WSecondaryTourneyTeams = pd.read_csv(join(data_path, 'WSecondaryTourneyTeams.csv'))
WSecondaryTourneyCompactResults = pd.read_csv(join(data_path, 'WSecondaryTourneyCompactResults.csv'))
WTeamSpellings = pd.read_csv(join(data_path, 'WTeamSpellings.csv'), encoding='cp1252')
WNCAATourneySlots = pd.read_csv(join(data_path, 'WNCAATourneySlots.csv'))

## Data preparation pipeline

In [3]:
def prepare_data(df):
        
    """
    This function duplicates and flips a game record.
    Now two records with the same data are present, 
    but viewed from perspectives of two different teams.
    """
    
    df = df[[
         'Season', 'DayNum', 'NumOT',
         'WTeamID',  'WScore', 'WLoc',
         'WFGM', 'WFGA', 'WFGM3', 'WFGA3', 'WFTM', 'WFTA', 'WOR', 'WDR', 'WAst', 'WTO', 'WStl', 'WBlk', 'WPF',
         'LTeamID', 'LScore',
         'LFGM', 'LFGA', 'LFGM3', 'LFGA3', 'LFTM', 'LFTA', 'LOR', 'LDR', 'LAst', 'LTO', 'LStl', 'LBlk', 'LPF'
    ]]
    dfswap = df[[
         'Season', 'DayNum', 'NumOT',
         'LTeamID', 'LScore', 'WLoc',
         'LFGM', 'LFGA', 'LFGM3', 'LFGA3', 'LFTM', 'LFTA', 'LOR', 'LDR', 'LAst', 'LTO', 'LStl', 'LBlk', 'LPF',
         'WTeamID',  'WScore',
         'WFGM', 'WFGA', 'WFGM3', 'WFGA3', 'WFTM', 'WFTA', 'WOR', 'WDR', 'WAst', 'WTO', 'WStl', 'WBlk', 'WPF',
    ]].copy()
    
    dfswap.loc[df['WLoc'] == 'H', 'WLoc'] = 'A'
    dfswap.loc[df['WLoc'] == 'A', 'WLoc'] = 'H'
        
    df = df.rename(columns={'WLoc': 'location'})
    dfswap = dfswap.rename(columns={'WLoc': 'location'})
        
    df.columns = [x.replace('W','T1_').replace('L','T2_') for x in list(df.columns)]
    dfswap.columns = [x.replace('L','T1_').replace('W','T2_') for x in list(dfswap.columns)]
    
    output = pd.concat([df, dfswap]).reset_index(drop=True)
    output.loc[output.location=='N','location'] = '0'
    output.loc[output.location=='H','location'] = '1'
    output.loc[output.location=='A','location'] = '-1'
    output.location = output.location.astype(int)
        
    output['PointDiff'] = output['T1_Score'] - output['T2_Score']
    
    return output

def get_data(regular_results, tourney_results, seeds, prepared=False, location_multiplier=[1, 1], win_ratio_days_back=14):

    """
    This function uses the prepare_data function in order to create
    a data frame with season statistics for each team.
    These statistics are added to records with games played in
    tournament to make data 'x' used in model to predict the game 
    result 'y'. The output is a data frame that contains data 'x'
    and also label 'y' can be easily calculated based on score difference in matches.
    """

    if prepared:
        regular_data = regular_results.copy()
        tourney_data = tourney_results.copy()
    else:
        # make data frames with extra rows to represent the perspective of losing team
        regular_data = prepare_data(regular_results)
        tourney_data = prepare_data(tourney_results)

    # ----------------------------------- Add reward/penalty for playing in home or away
    if location_multiplier[0] == 1 and location_multiplier[1] == 1:
        # data frame with mean game statistics for a given team in a given season
        season_statistics  = regular_data.groupby(["Season", 'T1_TeamID'])[
            [
                'T1_Score', 'T1_FGM','T1_FGA','T1_FGM3','T1_FGA3','T1_FTM','T1_FTA',
                'T1_OR','T1_DR','T1_Ast','T1_TO','T1_Stl','T1_Blk','T1_PF',
                'T2_Score', 'T2_FGM','T2_FGA','T2_FGM3','T2_FGA3','T2_FTM','T2_FTA',
                'T2_OR','T2_DR','T2_Ast','T2_TO','T2_Stl','T2_Blk','T2_PF',
                'PointDiff'
            ]
        ].agg('mean').reset_index()
    else:
        # Define which columns to adjust (you can add or remove columns as needed)
        T1_cols = ['T1_Score','T1_FGM','T1_FGA','T1_FGM3','T1_FGA3','T1_FTM','T1_FTA','T1_OR','T1_DR','T1_Ast','T1_TO','T1_Stl','T1_Blk','T1_PF']
        T2_cols = ['T2_Score','T2_FGM','T2_FGA','T2_FGM3','T2_FGA3','T2_FTM','T2_FTA','T2_OR','T2_DR','T2_Ast','T2_TO','T2_Stl','T2_Blk','T2_PF']
    
        # Convert the relevant columns to float before applying the adjustment function.
        cols_to_float = T1_cols + T2_cols
        regular_data[cols_to_float] = regular_data[cols_to_float].astype(float)
        
        def adjust_stats(row):
            # Determine multipliers based on location
            if row['location'] == 1:
                factor_T1 = location_multiplier[0]  # penalize Team1 stats (home)
                factor_T2 = location_multiplier[1]  # boost Team2 stats
            elif row['location'] == -1:
                factor_T1 = location_multiplier[1]  # boost Team1 stats (away)
                factor_T2 = location_multiplier[0]  # penalize Team2 stats
            else:
                factor_T1 = 1.0
                factor_T2 = 1.0
        
            # Adjust Team1 stats
            for col in T1_cols:
                if col in row and pd.notnull(row[col]):
                    row[col] = row[col] * factor_T1
        
            # Adjust Team2 stats
            for col in T2_cols:
                if col in row and pd.notnull(row[col]):
                    row[col] = row[col] * factor_T2
        
            # Recalculate derived statistics (if needed)
            if 'T1_Score' in row and 'T2_Score' in row:
                row['PointDiff'] = row['T1_Score'] - row['T2_Score']
            return row
        
        # Apply the adjustment function row-wise.
        regular_data_adjusted = regular_data.apply(adjust_stats, axis=1)
        
        # Now group by Season and T1_TeamID to compute season averages for the adjusted statistics.
        stats_columns = T1_cols[1:] + T2_cols[1:] + ['PointDiff']  # Exclude T1_TeamID from stats if present.
        season_statistics = regular_data_adjusted.groupby(["Season", 'T1_TeamID'])[stats_columns].agg('mean').reset_index()
    # -----------------------------------
    
    # mean statistics for team and team's opponent's
    season_statistics_T1 = season_statistics.copy()
    season_statistics_T2 = season_statistics.copy()
    
    season_statistics_T1.columns = ["T1_" + x.replace("T1_","").replace("T2_","opponent_") for x in list(season_statistics_T1.columns)]
    season_statistics_T2.columns = ["T2_" + x.replace("T1_","").replace("T2_","opponent_") for x in list(season_statistics_T2.columns)]
    season_statistics_T1.columns.values[0] = "Season"
    season_statistics_T2.columns.values[0] = "Season"
    season_statistics_T1 = season_statistics_T1.rename(columns={'T1_Score': 'T1_Score_mean'})
    season_statistics_T2 = season_statistics_T2.rename(columns={'T2_Score': 'T2_Score_mean'})
    
    # data frame containing game's result
    tourney_data = tourney_data[['Season', 'DayNum', 'T1_TeamID', 'T1_Score', 'T2_TeamID' ,'T2_Score', 'location']]
    tourney_data = pd.merge(tourney_data, season_statistics_T1, on = ['Season', 'T1_TeamID'], how = 'left')
    tourney_data = pd.merge(tourney_data, season_statistics_T2, on = ['Season', 'T2_TeamID'], how = 'left')

    calculate_win_ratio_days_back = 132 - win_ratio_days_back
    
    # data frame with win fraction from last x days for a given team in a given season
    last14days_stats_T1 = regular_data.loc[regular_data.DayNum>calculate_win_ratio_days_back].reset_index(drop=True)
    last14days_stats_T1['win'] = np.where(last14days_stats_T1['PointDiff']>0,1,0)
    last14days_stats_T1 = last14days_stats_T1.groupby(['Season','T1_TeamID'])['win'].mean().reset_index(name='T1_win_ratio_14d')
    
    last14days_stats_T2 = regular_data.loc[regular_data.DayNum>calculate_win_ratio_days_back].reset_index(drop=True)
    last14days_stats_T2['win'] = np.where(last14days_stats_T2['PointDiff']<0,1,0)
    last14days_stats_T2 = last14days_stats_T2.groupby(['Season','T2_TeamID'])['win'].mean().reset_index(name='T2_win_ratio_14d')
    
    # add to tourney_data column with win fraction for winning and losing team
    tourney_data = pd.merge(tourney_data, last14days_stats_T1, on = ['Season', 'T1_TeamID'], how = 'left')
    tourney_data = pd.merge(tourney_data, last14days_stats_T2, on = ['Season', 'T2_TeamID'], how = 'left')
    
    # get seeds with no regional division
    seeds['seed'] = seeds['Seed'].apply(lambda x: int(x[1:3]))
    
    # give each team a raw seed
    seeds_T1 = seeds[['Season','TeamID','seed']].copy()
    seeds_T2 = seeds[['Season','TeamID','seed']].copy()
    seeds_T1.columns = ['Season','T1_TeamID','T1_seed']
    seeds_T2.columns = ['Season','T2_TeamID','T2_seed']
    
    # add seeds to turney data for team 1 and team 2
    tourney_data = pd.merge(tourney_data, seeds_T1, on = ['Season', 'T1_TeamID'], how = 'left')
    tourney_data = pd.merge(tourney_data, seeds_T2, on = ['Season', 'T2_TeamID'], how = 'left')
    
    # add a seed difference column
    tourney_data["Seed_diff"] = tourney_data["T1_seed"] - tourney_data["T2_seed"]

    return tourney_data

def get_df(seeds,
            season_games, season_range, days_back,
            tourney_games, tourney_range, 
            location_multiplier=[1, 1],
           win_ratio_days_back=14):

    """
    This function uses get_data function to get a data
    frame which is then used to make 'x' and 'y' data
    used in models.
    """
    
    # Get from regular season games from specified season range and days back
    # The results of those games will be used as team statictics (additional team information for tourney games)
    regular_results = season_games[season_games['Season'].isin(season_range)]
    regular_results = regular_results[regular_results['DayNum'] > 134 - days_back]
    
    # Get tourney games from a specified season
    # The results of those games will be used as labels
    tourney_results = tourney_games[tourney_games['Season'].isin(tourney_range)]
    
    # Get final data frame
    df = get_data(regular_results, tourney_results, seeds, location_multiplier=[1, 1], win_ratio_days_back=win_ratio_days_back)
    
    return df

def get_final_df(seeds,
                   season_games, season_range, days_back,
                   tourney_games, tourney_range, 
                   SampleSubmissionStage1,
                  location_multiplier=[1, 1],
                win_ratio_days_back=14):

    """
    This function works the same as function get_x_y,
    but also creates (at the moment it's the same as get_x_y)
    aditional data points mostly with NaN values that matches
    the submission format (parsed team matchups with the sample submission file).
    """

    # Get from regular season games from specified season range and days back
    # The results of those games will be used as team statictics (additional team information for tourney games)
    regular_results = season_games[season_games['Season'].isin(season_range)]
    regular_results = regular_results[regular_results['DayNum'] > 134 - days_back]

    # Get tourney games from a specified season
    # The results of those games will be used as labels
    tourney_results = tourney_games[tourney_games['Season'].isin(tourney_range)]
    
    # Assume sample_submission is a DataFrame with an "ID" column like "2023_1101_1102"
    # and tourney_results is a DataFrame with columns including: Season, WTeamID, LTeamID, DayNum, WScore, LScore, WLoc, etc.
    # Filter rows where the ID starts with the specified season (followed by an underscore)
    final_season = tourney_range[0]
    sample_submission_copy = SampleSubmissionStage1.copy()
    sample_submission = sample_submission_copy[sample_submission_copy['ID'].str.startswith(f"{final_season}_")]
    sample_submission = sample_submission.drop(columns=['Pred'])
    
    sample_submission[['Season', 'Team1', 'Team2']] = sample_submission['ID'].str.split('_', expand=True)
    sample_submission['Season'] = sample_submission['Season'].astype(float)
    sample_submission['Team1'] = sample_submission['Team1'].astype(float)
    sample_submission['Team2'] = sample_submission['Team2'].astype(float)
    
    regular_data_final = prepare_data(regular_results)
    tourney_data_final  = prepare_data(tourney_results)
    
    tourney_data =  get_data(regular_data_final, tourney_data_final,
                             seeds, prepared=True,
                          win_ratio_days_back=win_ratio_days_back)
    tourney_data = pd.merge(
        sample_submission,
        tourney_data,
        left_on=['Season', 'Team1', 'Team2'],
        right_on=['Season', 'T1_TeamID', 'T2_TeamID'],
        how='left'
    )
    tourney_data['T1_TeamID'] = tourney_data['Team1']
    tourney_data['T2_TeamID'] = tourney_data['Team2']
    tourney_data = tourney_data.drop(['ID', 'Team1', 'Team2'], axis=1)
    
    return tourney_data

def correct_predictions_based_on_seed(x, y, maximum_favoured_seed = 4, number_of_added_columns=1):
    """
    Sets the winnning chance to 1 or 0 based on the seed difference.
    """
    for i in range(len(x)):
        if x[i][-1-number_of_added_columns] <= (-16 + maximum_favoured_seed * 2 - 1):
            y[i] = 1
        elif x[i][-1-number_of_added_columns] >= (16 - maximum_favoured_seed * 2 + 1):
            y[i] = 0
    return y

def clear_na_from_x_y(x, y):
    """
    The data frame for final season is in format matching the submission file.
    This function clears NaNs from data.
    """
    # Create masks for training data:
    mask_train = ~np.isnan(x).any(axis=1) & ~np.isnan(y)
    x_clean = x[mask_train]
    y_clean = y[mask_train]
    return x_clean, y_clean

def x_y_from_data_frame(df):
    # Prepare data and labels
    x = df[list(df.columns[7:])].values
    y = np.where(
        df[['T1_Score', 'T2_Score']].isnull().any(axis=1),
        np.nan,
        np.where(df['T1_Score'] - df['T2_Score'] > 0, 1, 0)
    )
    return x, y

def get_all_core_data(
    regular_results, tourney_results, seeds, SampleSubmissionStage1,
    final_season = 2024, # the season we want to predict, so for out submission it will be 2025
    start_season = 2005, # from which ponit should we begin creating data
    season_years_list  = [[i-1, i] for i in range(2005, 2024+1)], # at which seasons to look at when calculating team's stats
    days_back = 15, # how many days back from the start of tourney to calculate team's stats per season
    maximum_favoured_seed = 4, # Set the predicted probability of winning to 1 for seeds <= maximum_favoured_seed and to 0 for >= 16-maximum_favoured_seed
    location_multiplier=[0.95, 1.05], # home penalty, away bonus 
    include_men = True, # include M... data sets when preparing x and y
    include_women = True, # include W... data sets when preparing x and y
    win_ratio_days_back = 14):

    """
    The idea of this function is to easily get data needed to train and test the model later on, with minimal code
    to not clutter the netebook.
    
    This function outputs df, x, y, df_final, x_final_season, y_final_season.
    
    df is a data frame with first 6 columns from tourney games and other calculated from other data frames.
    
    Adding a column to df and executing x_y_from_data_frame(df) function will yield x with added data.
    
    Note that df_final has the same structure as df, but also with rows with NaNs. The rows with missing information are there
    to match the sumbission file format. The separation of those data frames is to ensure that information from last season doesn't
    leak into training data due to poorly written code.
    
    x has df columns from location onwardsthe columns before that are from tourney games and are used to calculate y (based on points).
    
    y has label 0 or 1 (lose or win) and nan if the correspoinding data in x was nan.
    
    There is also x_final_season and y_final_season aquired from df_final, which are the same as x and y, but like df_final, they have NaNs. 
    """

    # This ensures that we simulate the scenario in competition
    tourney_results_final = tourney_results[tourney_results['Season'] == final_season]
    tourney_results = tourney_results[tourney_results['Season'] < final_season]
    regular_results = regular_results[regular_results['Season'] != 2020] # This year had no tournament data
    tourney_years_list = [[i] for i in range(start_season, final_season+1)]
    
    # Arrays to store data
    data = []
    data_final = []
    for season_years, tourney_years in zip(season_years_list, tourney_years_list):
            
        # Create data separately for the of games
        # The separation is to ensure there is no data leak
        if tourney_years[0] == final_season:
            data_tmp = get_df(
                seeds,
                regular_results, season_years, days_back,
                tourney_results_final, tourney_years,
                location_multiplier=location_multiplier,
                win_ratio_days_back=win_ratio_days_back
            )
            data_final.append(data_tmp)
        else:
            data_tmp = get_df(
                seeds,
                regular_results, season_years, days_back,
                tourney_results, tourney_years,
                location_multiplier=location_multiplier,
                win_ratio_days_back=win_ratio_days_back
            )
            data.append(data_tmp)
            
    df = pd.concat(data, ignore_index=True)
    df_final = pd.concat(data_final, ignore_index=True)
    
    return df, df_final

def brier_for_all_years(year_range, season_years_list):
    
    df_train_women_list = []
    df_test_women_list = []
    df_train_men_list = []
    df_test_men_list = []
    
    for year, season_years in zip(year_range, season_years_list):
    
        final_season = year

        for sex in ['woman', 'man']:
            
            if sex == 'woman':
                include_men = False
                include_women = True
            elif sex == 'man':
                include_men = True
                include_women = False

            # ----------------------------------------------------------
            # READ DATA
            regular_results = pd.concat([
                MRegularSeasonDetailedResults.copy() if include_men else None,
                WRegularSeasonDetailedResults.copy() if include_women else None
            ], ignore_index=True)
            tourney_results = pd.concat([
                MNCAATourneyDetailedResults.copy() if include_men else None,
                WNCAATourneyDetailedResults.copy() if include_women else None
            ], ignore_index=True)
            seeds = pd.concat([
                MNCAATourneySeeds.copy() if include_men else None,
                WNCAATourneySeeds.copy() if include_women else None
            ], ignore_index=True)
            # ----------------------------------------------------------
            # GET ALL DATA NEEDED TO USE THE MODELS
            df_train, df_test = get_all_core_data(
                regular_results, tourney_results, seeds, SampleSubmissionStage1, final_season = final_season,
                start_season = start_season, season_years_list  = season_years, days_back = days_back,
                maximum_favoured_seed = maximum_favoured_seed, location_multiplier=location_multiplier,
                include_men = include_men, include_women = include_women, win_ratio_days_back = win_ratio_days_back)
            # ----------------------------------------------------------
            # ADD TEAM AND COACH ELO AND REPLACE NAN WITH MEAN
            elo = pd.read_csv(join(data_path, 'elo.csv'))
            elo['CoachELO'] = elo['CoachELO'].fillna(elo['CoachELO'].mean())
            elo = elo.drop(['CoachName'], axis=1)
            def add_elo_column(df):
                df = df.copy()
                df = pd.merge(
                        df,
                        elo[['Season', 'DayNum', 'TeamID', 'TeamELO', 'CoachELO']],
                        left_on=['Season', 'DayNum', 'T1_TeamID'],
                        right_on=['Season', 'DayNum', 'TeamID'],
                        how='left'
                    )
                df = df.drop(['TeamID'], axis=1)
                return df
            df_train = add_elo_column(df_train)
            df_test = add_elo_column(df_test)
            # ----------------------------------------------------------
            if sex == 'woman':
                df_train_women_list.append(df_train.copy())
                df_test_women_list.append(df_test.copy())
            elif sex == 'man':
                df_train_men_list.append(df_train.copy())
                df_test_men_list.append(df_test.copy())
                
        for i in range(len(df_train_men_list)):
            df_train_women_list[i] = df_train_women_list[i][list(df_train_men_list[0])]
            df_test_women_list[i] = df_test_women_list[i][list(df_train_men_list[0])]
            
    return df_train_women_list, df_test_women_list, df_train_men_list, df_test_men_list

## Get data to use models on

In [38]:
columns_to_include_women = [
 'Season',
 'DayNum',
 'T1_TeamID',
 'T1_Score',
 'T2_TeamID',
 'T2_Score',
 'location',
                      
#  'T1_Score_mean',
 'T1_FGM',
 'T1_FGA',
 'T1_FGM3',
 'T1_FGA3',
#  'T1_FTM',
#  'T1_FTA',
 'T1_OR',
#  'T1_DR',
 'T1_Ast',
 'T1_TO',
 'T1_Stl',
#  'T1_Blk',
 'T1_PF',
                      
#  'T1_opponent_Score',
 'T1_opponent_FGM',
 'T1_opponent_FGA',
 'T1_opponent_FGM3',
 'T1_opponent_FGA3',
#  'T1_opponent_FTM',
#  'T1_opponent_FTA',
 'T1_opponent_OR',
#  'T1_opponent_DR',
 'T1_opponent_Ast',
 'T1_opponent_TO',
 'T1_opponent_Stl',
#  'T1_opponent_Blk',
 'T1_opponent_PF',
 'T1_PointDiff',
                      
#  'T2_Score_mean',
 'T2_FGM',
 'T2_FGA',
 'T2_FGM3',
 'T2_FGA3',
#  'T2_FTM',
#  'T2_FTA',
 'T2_OR',
#  'T2_DR',
 'T2_Ast',
 'T2_TO',
 'T2_Stl',
#  'T2_Blk',
 'T2_PF',
                      
#  'T2_opponent_Score',
 'T2_opponent_FGM',
 'T2_opponent_FGA',
 'T2_opponent_FGM3',
 'T2_opponent_FGA3',
#  'T2_opponent_FTM',
#  'T2_opponent_FTA',
 'T2_opponent_OR',
#  'T2_opponent_DR',
 'T2_opponent_Ast',
 'T2_opponent_TO',
 'T2_opponent_Stl',
#  'T2_opponent_Blk',
 'T2_opponent_PF',
 'T2_PointDiff',
                      
 'T1_win_ratio_14d',
 'T2_win_ratio_14d',
 'T1_seed',
 'T2_seed',
 'Seed_diff',
 'TeamELO',
#  'CoachELO'
]
columns_to_include_men = [
 'Season',
 'DayNum',
 'T1_TeamID',
 'T1_Score',
 'T2_TeamID',
 'T2_Score',
 'location',
                      
#  'T1_Score_mean',
 'T1_FGM',
 'T1_FGA',
 'T1_FGM3',
 'T1_FGA3',
#  'T1_FTM',
#  'T1_FTA',
 'T1_OR',
#  'T1_DR',
 'T1_Ast',
 'T1_TO',
 'T1_Stl',
#  'T1_Blk',
 'T1_PF',
                      
#  'T1_opponent_Score',
 'T1_opponent_FGM',
 'T1_opponent_FGA',
 'T1_opponent_FGM3',
 'T1_opponent_FGA3',
#  'T1_opponent_FTM',
#  'T1_opponent_FTA',
 'T1_opponent_OR',
#  'T1_opponent_DR',
 'T1_opponent_Ast',
 'T1_opponent_TO',
 'T1_opponent_Stl',
#  'T1_opponent_Blk',
 'T1_opponent_PF',
 'T1_PointDiff',
                      
#  'T2_Score_mean',
 'T2_FGM',
 'T2_FGA',
 'T2_FGM3',
 'T2_FGA3',
#  'T2_FTM',
#  'T2_FTA',
 'T2_OR',
#  'T2_DR',
 'T2_Ast',
 'T2_TO',
 'T2_Stl',
#  'T2_Blk',
 'T2_PF',
                      
#  'T2_opponent_Score',
 'T2_opponent_FGM',
 'T2_opponent_FGA',
 'T2_opponent_FGM3',
 'T2_opponent_FGA3',
#  'T2_opponent_FTM',
#  'T2_opponent_FTA',
 'T2_opponent_OR',
#  'T2_opponent_DR',
 'T2_opponent_Ast',
 'T2_opponent_TO',
 'T2_opponent_Stl',
#  'T2_opponent_Blk',
 'T2_opponent_PF',
 'T2_PointDiff',
                      
 'T1_win_ratio_14d',
 'T2_win_ratio_14d',
 'T1_seed',
 'T2_seed',
 'Seed_diff',
 'TeamELO',
 'CoachELO'
]

year_range = [2023]
start_season = 2003 # from which ponit should we begin creating data
season_years_list  = [[[i] for i in range(start_season, year+1)] for year in year_range] # at which seasons to look at when calculating team's stats
days_back = 25 # how many days back from the start of tourney to calculate team's stats per season
location_multiplier = [0.95, 1.05] # home penalty, away bonus ex.
win_ratio_days_back = 14 # how many days back from the tourney do we calculate win ratio
maximum_favoured_seed = 0
# -------------------------------------------
df_train_women_list, df_test_women_list, df_train_men_list, df_test_men_list = brier_for_all_years(year_range, season_years_list)
x_train_women_list = []
x_test_women_list = []
x_train_men_list = []
x_test_men_list = []
y_train_women_list = []
y_test_women_list = []
y_train_men_list = []
y_test_men_list = []
for i in range(len(year_range)):

    if len(year_range) > 1:
        df_train_women_list[i] = df_train_women_list[i][columns_to_include_women]
        df_test_women_list[i] = df_test_women_list[i][columns_to_include_women]
        df_train_men_list[i] = df_train_men_list[i][columns_to_include_men]
        df_test_men_list[i] = df_test_men_list[i][columns_to_include_men]
        
        x_train_women, y_train_women = x_y_from_data_frame(df_train_women_list[i])
        x_test_women, y_test_women = x_y_from_data_frame(df_test_women_list[i])
        x_train_men, y_train_men = x_y_from_data_frame(df_train_men_list[i])
        x_test_men, y_test_men = x_y_from_data_frame(df_test_men_list[i])

        x_train_women, y_train_women = clear_na_from_x_y(x_train_women.copy(), y_train_women.copy())
        x_test_women, y_test_women = clear_na_from_x_y(x_test_women.copy(), y_test_women.copy())
        x_train_men, y_train_men = clear_na_from_x_y(x_train_men.copy(), y_train_men.copy())
        x_test_men, y_test_men = clear_na_from_x_y(x_test_men.copy(), y_test_men.copy())
        
        x_train_women_list.append(x_train_women)
        x_test_women_list.append(x_test_women)
        x_train_men_list.append(x_train_men)
        x_test_men_list.append(x_test_men)

        y_train_women_list.append(y_train_women)
        y_test_women_list.append(y_test_women)
        y_train_men_list.append(y_train_men)
        y_test_men_list.append(y_test_men)
    
    else:
        df_train_women_list = df_train_women_list[0][columns_to_include_women]
        df_test_women_list = df_test_women_list[0][columns_to_include_women]
        df_train_men_list = df_train_men_list[0][columns_to_include_men]
        df_test_men_list = df_test_men_list[0][columns_to_include_men]

        x_train_women, y_train_women = x_y_from_data_frame(df_train_women_list)
        x_test_women, y_test_women = x_y_from_data_frame(df_test_women_list)
        x_train_men, y_train_men = x_y_from_data_frame(df_train_men_list)
        x_test_men, y_test_men = x_y_from_data_frame(df_test_men_list)

        x_train_women, y_train_women = clear_na_from_x_y(x_train_women.copy(), y_train_women.copy())
        x_test_women, y_test_women = clear_na_from_x_y(x_test_women.copy(), y_test_women.copy())
        x_train_men, y_train_men = clear_na_from_x_y(x_train_men.copy(), y_train_men.copy())
        x_test_men, y_test_men = clear_na_from_x_y(x_test_men.copy(), y_test_men.copy())

## Which models to consider

In [5]:
def train_and_evaluate(model, x_train, y_train, x_test, y_test):
    model.fit(x_train, y_train)
    y_pred = model.predict_proba(x_test)[:, 1]  # Get probability of class 1
    score = brier_score_loss(y_test, y_pred)
    return score

# Initialize models
catboost_model = CatBoostClassifier(verbose=0, iterations=500, depth=6, learning_rate=0.05)
xgboost_model = XGBClassifier(n_estimators=500, max_depth=6, learning_rate=0.05, use_label_encoder=False, eval_metric='logloss')
mlp_model = MLPClassifier(hidden_layer_sizes=(100, 50), max_iter=500, alpha=0.01)
lr_model = LogisticRegression()

# Train and evaluate for women
y_pred_women_catboost = train_and_evaluate(catboost_model, x_train_women, y_train_women, x_test_women, y_test_women)
y_pred_women_xgboost = train_and_evaluate(xgboost_model, x_train_women, y_train_women, x_test_women, y_test_women)
y_pred_women_mlp = train_and_evaluate(mlp_model, x_train_women, y_train_women, x_test_women, y_test_women)
y_pred_women_lr = train_and_evaluate(lr_model, x_train_women, y_train_women, x_test_women, y_test_women)

# Train and evaluate for men
y_pred_men_catboost = train_and_evaluate(catboost_model, x_train_men, y_train_men, x_test_men, y_test_men)
y_pred_men_xgboost = train_and_evaluate(xgboost_model, x_train_men, y_train_men, x_test_men, y_test_men)
y_pred_men_mlp = train_and_evaluate(mlp_model, x_train_men, y_train_men, x_test_men, y_test_men)
y_pred_men_lr = train_and_evaluate(lr_model, x_train_men, y_train_men, x_test_men, y_test_men)

print("Brier Scores:")
print(f"CatBoost (Women): {y_pred_women_catboost}")
print(f"XGBoost (Women): {y_pred_women_xgboost}")
print(f"MLP (Women): {y_pred_women_mlp}")
print(f"Logistic (Women): {y_pred_women_lr}")
print(f"CatBoost (Men): {y_pred_men_catboost}")
print(f"XGBoost (Men): {y_pred_men_xgboost}")
print(f"MLP (Men): {y_pred_men_mlp}")
print(f"Logistic (Men): {y_pred_men_lr}")

C:\Users\Sebastian\anaconda3\lib\site-packages\sklearn\linear_model\_logistic.py:814: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. of ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


Brier Scores:
CatBoost (Women): 0.19789686715342686
XGBoost (Women): 0.21424128811938234
MLP (Women): 0.24739413922799655
Logistic (Women): 0.1612005025608674
CatBoost (Men): 0.22776204616428086
XGBoost (Men): 0.23156317233882792
MLP (Men): 0.3434688633657514
Logistic (Men): 0.20392680092784002


C:\Users\Sebastian\anaconda3\lib\site-packages\sklearn\linear_model\_logistic.py:814: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. of ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


# Training models

## CatBoost

In [6]:
# FOR WOMEN
def objective(trial, x_train, y_train, x_val, y_val):
    params = {
        "iterations": trial.suggest_int("iterations", 500, 3000),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.1, log=True),
        "depth": trial.suggest_int("depth", 4, 10),
        "l2_leaf_reg": trial.suggest_float("l2_leaf_reg", 1, 20, log=True),
        "border_count": trial.suggest_int("border_count", 32, 255),
        "random_strength": trial.suggest_float("random_strength", 1e-3, 10, log=True),
        "bagging_temperature": trial.suggest_float("bagging_temperature", 0.0, 1.0),
        "verbose": 0,
        "loss_function": "Logloss"
    }
    model = CatBoostClassifier(**params)
    model.fit(x_train, y_train, eval_set=(x_val, y_val), early_stopping_rounds=50, verbose=False)
    
    y_pred = model.predict_proba(x_val)[:, 1]
    return brier_score_loss(y_val, y_pred)

###################################################### WOMEN

# Split data into training and validation for hyperparameter tuning
x_train_women_tr, x_val_women, y_train_women_tr, y_val_women = train_test_split(x_train_women, y_train_women, test_size=0.2, random_state=42)

# Run optimization
study = optuna.create_study(direction="minimize")
study.optimize(lambda trial: objective(trial, x_train_women_tr, y_train_women_tr, x_val_women, y_val_women), n_trials=50)

# Train final model with best hyperparameters
best_params_women = study.best_params
best_catboost_women = CatBoostClassifier(**best_params_women)
best_catboost_women.fit(x_train_women, y_train_women)

# Predict on test set
y_pred_women = best_catboost_women.predict_proba(x_test_women)[:, 1]

# Compute Brier Score
brier_women = brier_score_loss(y_test_women, y_pred_women)
print("Best CatBoost Brier Score (Women):", brier_women)

###################################################### MEN

# Split data into training and validation for hyperparameter tuning
x_train_men_tr, x_val_men, y_train_men_tr, y_val_men = train_test_split(x_train_men, y_train_men, test_size=0.2, random_state=42)

# Run optimization
study = optuna.create_study(direction="minimize")
study.optimize(lambda trial: objective(trial, x_train_men_tr, y_train_men_tr, x_val_men, y_val_men), n_trials=50)

# Train final model with best hyperparameters
best_params_men = study.best_params
best_catboost_men = CatBoostClassifier(**best_params_men)
best_catboost_men.fit(x_train_men, y_train_men)

# Predict on test set
y_pred_men = best_catboost_men.predict_proba(x_test_men)[:, 1]

# Compute Brier Score
brier_men = brier_score_loss(y_test_men, y_pred_men)
print("Best CatBoost Brier Score (men):", brier_men)

[I 2025-03-19 20:43:51,040] A new study created in memory with name: no-name-feb760ed-5d1f-4384-8de8-67a831731c92
[I 2025-03-19 20:43:54,302] Trial 0 finished with value: 0.1406591632491876 and parameters: {'iterations': 1390, 'learning_rate': 0.01621631015665751, 'depth': 8, 'l2_leaf_reg': 1.5181430318588198, 'border_count': 92, 'random_strength': 0.07730240352213018, 'bagging_temperature': 0.3331595511496629}. Best is trial 0 with value: 0.1406591632491876.
[I 2025-03-19 20:43:56,410] Trial 1 finished with value: 0.13612393715579663 and parameters: {'iterations': 2161, 'learning_rate': 0.019816243512429128, 'depth': 5, 'l2_leaf_reg': 1.353003014251472, 'border_count': 235, 'random_strength': 0.5867927574514905, 'bagging_temperature': 0.9006794043561132}. Best is trial 1 with value: 0.13612393715579663.
[I 2025-03-19 20:43:58,540] Trial 2 finished with value: 0.1390262429636072 and parameters: {'iterations': 2974, 'learning_rate': 0.016505066663263044, 'depth': 5, 'l2_leaf_reg': 13.71

[I 2025-03-19 20:44:43,698] Trial 23 finished with value: 0.138367534183508 and parameters: {'iterations': 1643, 'learning_rate': 0.07598980567361457, 'depth': 5, 'l2_leaf_reg': 2.210297410087406, 'border_count': 254, 'random_strength': 0.1394539640194164, 'bagging_temperature': 0.8279081382281566}. Best is trial 21 with value: 0.13342173920710473.
[I 2025-03-19 20:44:44,766] Trial 24 finished with value: 0.13723764024076915 and parameters: {'iterations': 2360, 'learning_rate': 0.05582986472702592, 'depth': 4, 'l2_leaf_reg': 3.3743229776086214, 'border_count': 155, 'random_strength': 1.344568237376433, 'bagging_temperature': 0.9650452772235305}. Best is trial 21 with value: 0.13342173920710473.
[I 2025-03-19 20:44:46,052] Trial 25 finished with value: 0.13861160645106224 and parameters: {'iterations': 1967, 'learning_rate': 0.029646186411054643, 'depth': 5, 'l2_leaf_reg': 6.298246577199194, 'border_count': 217, 'random_strength': 0.021218611822496444, 'bagging_temperature': 0.691531877

[I 2025-03-19 20:45:32,276] Trial 47 finished with value: 0.13874745134901265 and parameters: {'iterations': 2110, 'learning_rate': 0.021189630653744417, 'depth': 4, 'l2_leaf_reg': 9.456259826155403, 'border_count': 245, 'random_strength': 0.017605591181847863, 'bagging_temperature': 0.8428551668211369}. Best is trial 21 with value: 0.13342173920710473.
[I 2025-03-19 20:45:34,886] Trial 48 finished with value: 0.1386285669419121 and parameters: {'iterations': 1854, 'learning_rate': 0.013083788716575617, 'depth': 5, 'l2_leaf_reg': 1.8628508008590636, 'border_count': 223, 'random_strength': 0.8360946958754758, 'bagging_temperature': 0.6700639863901983}. Best is trial 21 with value: 0.13342173920710473.
[I 2025-03-19 20:45:38,034] Trial 49 finished with value: 0.1376644825497981 and parameters: {'iterations': 1625, 'learning_rate': 0.015314059747427645, 'depth': 5, 'l2_leaf_reg': 1.0022525389187555, 'border_count': 234, 'random_strength': 3.3165816471190617, 'bagging_temperature': 0.94481

0:	learn: 0.6815818	total: 5.65ms	remaining: 11s
1:	learn: 0.6656314	total: 10.7ms	remaining: 10.5s
2:	learn: 0.6547847	total: 15.9ms	remaining: 10.3s
3:	learn: 0.6437125	total: 20.9ms	remaining: 10.2s
4:	learn: 0.6366559	total: 26ms	remaining: 10.1s
5:	learn: 0.6267863	total: 31ms	remaining: 10s
6:	learn: 0.6162972	total: 35.9ms	remaining: 9.96s
7:	learn: 0.6073977	total: 41ms	remaining: 9.96s
8:	learn: 0.5992786	total: 46ms	remaining: 9.91s
9:	learn: 0.5921237	total: 51.2ms	remaining: 9.94s
10:	learn: 0.5866038	total: 56.1ms	remaining: 9.89s
11:	learn: 0.5803256	total: 61.1ms	remaining: 9.86s
12:	learn: 0.5748328	total: 66ms	remaining: 9.84s
13:	learn: 0.5693353	total: 71.2ms	remaining: 9.84s
14:	learn: 0.5625671	total: 76.1ms	remaining: 9.82s
15:	learn: 0.5563301	total: 81ms	remaining: 9.8s
16:	learn: 0.5519265	total: 86.1ms	remaining: 9.79s
17:	learn: 0.5472001	total: 91.2ms	remaining: 9.78s
18:	learn: 0.5427279	total: 96.2ms	remaining: 9.78s
19:	learn: 0.5380245	total: 101ms	remai

182:	learn: 0.3463678	total: 923ms	remaining: 8.91s
183:	learn: 0.3454101	total: 929ms	remaining: 8.91s
184:	learn: 0.3448097	total: 935ms	remaining: 8.92s
185:	learn: 0.3444909	total: 941ms	remaining: 8.92s
186:	learn: 0.3441353	total: 946ms	remaining: 8.92s
187:	learn: 0.3437308	total: 951ms	remaining: 8.92s
188:	learn: 0.3433775	total: 957ms	remaining: 8.91s
189:	learn: 0.3429188	total: 962ms	remaining: 8.91s
190:	learn: 0.3425159	total: 967ms	remaining: 8.91s
191:	learn: 0.3422840	total: 973ms	remaining: 8.9s
192:	learn: 0.3417824	total: 978ms	remaining: 8.9s
193:	learn: 0.3413788	total: 983ms	remaining: 8.9s
194:	learn: 0.3407703	total: 989ms	remaining: 8.9s
195:	learn: 0.3402228	total: 994ms	remaining: 8.9s
196:	learn: 0.3395724	total: 1000ms	remaining: 8.89s
197:	learn: 0.3390021	total: 1s	remaining: 8.89s
198:	learn: 0.3385275	total: 1.01s	remaining: 8.89s
199:	learn: 0.3380288	total: 1.01s	remaining: 8.88s
200:	learn: 0.3374765	total: 1.02s	remaining: 8.88s
201:	learn: 0.33697

...

789:	learn: 0.0422870	total: 1m 1s	remaining: 1m
790:	learn: 0.0422228	total: 1m 1s	remaining: 60s
791:	learn: 0.0421377	total: 1m 1s	remaining: 59.9s
792:	learn: 0.0420598	total: 1m 1s	remaining: 59.8s
793:	learn: 0.0419822	total: 1m 1s	remaining: 59.7s
794:	learn: 0.0418978	total: 1m 1s	remaining: 59.6s
795:	learn: 0.0418435	total: 1m 1s	remaining: 59.5s
796:	learn: 0.0417696	total: 1m 1s	remaining: 59.5s
797:	learn: 0.0416694	total: 1m 2s	remaining: 59.4s
798:	learn: 0.0415994	total: 1m 2s	remaining: 59.3s
799:	learn: 0.0415517	total: 1m 2s	remaining: 59.2s
800:	learn: 0.0415086	total: 1m 2s	remaining: 59.1s
801:	learn: 0.0414131	total: 1m 2s	remaining: 59s
802:	learn: 0.0413355	total: 1m 2s	remaining: 58.9s
803:	learn: 0.0412735	total: 1m 2s	remaining: 58.9s
804:	learn: 0.0412128	total: 1m 2s	remaining: 58.8s
805:	learn: 0.0411513	total: 1m 2s	remaining: 58.7s
806:	learn: 0.0410664	total: 1m 2s	remaining: 58.6s
807:	learn: 0.0409734	total: 1m 2s	remaining: 58.5s
808:	learn: 0.04090

947:	learn: 0.0324917	total: 1m 12s	remaining: 47.1s
948:	learn: 0.0324432	total: 1m 12s	remaining: 47s
949:	learn: 0.0323893	total: 1m 12s	remaining: 46.9s
950:	learn: 0.0323300	total: 1m 12s	remaining: 46.9s
951:	learn: 0.0322653	total: 1m 12s	remaining: 46.8s
952:	learn: 0.0322239	total: 1m 13s	remaining: 46.7s
953:	learn: 0.0321804	total: 1m 13s	remaining: 46.6s
954:	learn: 0.0321311	total: 1m 13s	remaining: 46.5s
955:	learn: 0.0320719	total: 1m 13s	remaining: 46.4s
956:	learn: 0.0320234	total: 1m 13s	remaining: 46.4s
957:	learn: 0.0319870	total: 1m 13s	remaining: 46.3s
958:	learn: 0.0319551	total: 1m 13s	remaining: 46.2s
959:	learn: 0.0319136	total: 1m 13s	remaining: 46.1s
960:	learn: 0.0318685	total: 1m 13s	remaining: 46s
961:	learn: 0.0318208	total: 1m 13s	remaining: 46s
962:	learn: 0.0317742	total: 1m 13s	remaining: 45.9s
963:	learn: 0.0317210	total: 1m 13s	remaining: 45.8s
964:	learn: 0.0316795	total: 1m 13s	remaining: 45.7s
965:	learn: 0.0316328	total: 1m 13s	remaining: 45.6s

1103:	learn: 0.0258856	total: 1m 23s	remaining: 34.8s
1104:	learn: 0.0258483	total: 1m 23s	remaining: 34.7s
1105:	learn: 0.0258122	total: 1m 23s	remaining: 34.6s
1106:	learn: 0.0257849	total: 1m 24s	remaining: 34.6s
1107:	learn: 0.0257425	total: 1m 24s	remaining: 34.5s
1108:	learn: 0.0257003	total: 1m 24s	remaining: 34.4s
1109:	learn: 0.0256673	total: 1m 24s	remaining: 34.3s
1110:	learn: 0.0256326	total: 1m 24s	remaining: 34.2s
1111:	learn: 0.0256020	total: 1m 24s	remaining: 34.2s
1112:	learn: 0.0255683	total: 1m 24s	remaining: 34.1s
1113:	learn: 0.0255377	total: 1m 24s	remaining: 34s
1114:	learn: 0.0255023	total: 1m 24s	remaining: 33.9s
1115:	learn: 0.0254613	total: 1m 24s	remaining: 33.9s
1116:	learn: 0.0254302	total: 1m 24s	remaining: 33.8s
1117:	learn: 0.0254068	total: 1m 24s	remaining: 33.7s
1118:	learn: 0.0253756	total: 1m 24s	remaining: 33.6s
1119:	learn: 0.0253452	total: 1m 25s	remaining: 33.5s
1120:	learn: 0.0253079	total: 1m 25s	remaining: 33.5s
1121:	learn: 0.0252770	total: 

1257:	learn: 0.0213589	total: 1m 35s	remaining: 23s
1258:	learn: 0.0213431	total: 1m 35s	remaining: 22.9s
1259:	learn: 0.0213225	total: 1m 35s	remaining: 22.8s
1260:	learn: 0.0212984	total: 1m 35s	remaining: 22.7s
1261:	learn: 0.0212816	total: 1m 35s	remaining: 22.7s
1262:	learn: 0.0212559	total: 1m 35s	remaining: 22.6s
1263:	learn: 0.0212274	total: 1m 35s	remaining: 22.5s
1264:	learn: 0.0211986	total: 1m 35s	remaining: 22.4s
1265:	learn: 0.0211742	total: 1m 35s	remaining: 22.4s
1266:	learn: 0.0211468	total: 1m 35s	remaining: 22.3s
1267:	learn: 0.0211223	total: 1m 35s	remaining: 22.2s
1268:	learn: 0.0210916	total: 1m 35s	remaining: 22.1s
1269:	learn: 0.0210629	total: 1m 35s	remaining: 22.1s
1270:	learn: 0.0210406	total: 1m 36s	remaining: 22s
1271:	learn: 0.0210136	total: 1m 36s	remaining: 21.9s
1272:	learn: 0.0209897	total: 1m 36s	remaining: 21.8s
1273:	learn: 0.0209620	total: 1m 36s	remaining: 21.8s
1274:	learn: 0.0209319	total: 1m 36s	remaining: 21.7s
1275:	learn: 0.0209118	total: 1m

1412:	learn: 0.0180413	total: 1m 46s	remaining: 11.2s
1413:	learn: 0.0180224	total: 1m 46s	remaining: 11.1s
1414:	learn: 0.0179991	total: 1m 46s	remaining: 11s
1415:	learn: 0.0179796	total: 1m 46s	remaining: 11s
1416:	learn: 0.0179615	total: 1m 46s	remaining: 10.9s
1417:	learn: 0.0179418	total: 1m 46s	remaining: 10.8s
1418:	learn: 0.0179237	total: 1m 46s	remaining: 10.7s
1419:	learn: 0.0179065	total: 1m 46s	remaining: 10.7s
1420:	learn: 0.0178928	total: 1m 46s	remaining: 10.6s
1421:	learn: 0.0178779	total: 1m 46s	remaining: 10.5s
1422:	learn: 0.0178597	total: 1m 46s	remaining: 10.4s
1423:	learn: 0.0178401	total: 1m 46s	remaining: 10.4s
1424:	learn: 0.0178197	total: 1m 47s	remaining: 10.3s
1425:	learn: 0.0177993	total: 1m 47s	remaining: 10.2s
1426:	learn: 0.0177782	total: 1m 47s	remaining: 10.1s
1427:	learn: 0.0177626	total: 1m 47s	remaining: 10.1s
1428:	learn: 0.0177480	total: 1m 47s	remaining: 9.99s
1429:	learn: 0.0177286	total: 1m 47s	remaining: 9.91s
1430:	learn: 0.0177068	total: 1m

## XGBoost

In [7]:
import optuna
from xgboost import XGBClassifier
from sklearn.metrics import brier_score_loss
from sklearn.model_selection import train_test_split

############################################
# FOR WOMEN
############################################
def objective_xgb(trial, x_train, y_train, x_val, y_val):
    params = {
        "n_estimators": trial.suggest_int("n_estimators", 100, 1000),
        "max_depth": trial.suggest_int("max_depth", 3, 10),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.1, log=True),
        "subsample": trial.suggest_float("subsample", 0.6, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.6, 1.0),
        "min_child_weight": trial.suggest_int("min_child_weight", 3, 20),
        "verbosity": 0
    }
    
    model = XGBClassifier(**params, use_label_encoder=False, eval_metric="logloss", early_stopping_rounds=50)
    model.fit(x_train, y_train, eval_set=[(x_val, y_val)], verbose=False)
    y_pred = model.predict_proba(x_val)[:, 1]
    return brier_score_loss(y_val, y_pred)

# Split women's training data for hyperparameter tuning
x_train_women_tr, x_val_women, y_train_women_tr, y_val_women = train_test_split(
    x_train_women, y_train_women, test_size=0.2, random_state=42
)

# Run optimization for women
study_women = optuna.create_study(direction="minimize")
study_women.optimize(lambda trial: objective_xgb(trial, x_train_women_tr, y_train_women_tr, x_val_women, y_val_women), n_trials=50)
best_params_women_xgb = study_women.best_params
print("Best XGBoost Params (Women):", best_params_women_xgb)

# Train final XGBoost model for women with best hyperparameters
best_xgb_women = XGBClassifier(**best_params_women_xgb, use_label_encoder=False, eval_metric="logloss")
best_xgb_women.fit(x_train_women, y_train_women)
y_pred_women_xgb = best_xgb_women.predict_proba(x_test_women)[:, 1]
brier_women_xgb = brier_score_loss(y_test_women, y_pred_women_xgb)
print("Best XGBoost Brier Score (Women):", brier_women_xgb)

############################################
# FOR MEN
############################################
def objective_xgb_men(trial, x_train, y_train, x_val, y_val):
    # We can use the same search space as for women
    params = {
        "n_estimators": trial.suggest_int("n_estimators", 100, 1000),
        "max_depth": trial.suggest_int("max_depth", 3, 10),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.1, log=True),
        "subsample": trial.suggest_float("subsample", 0.6, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.6, 1.0),
        "min_child_weight": trial.suggest_int("min_child_weight", 3, 20),
        "verbosity": 0
    }
    
    model = XGBClassifier(**params, use_label_encoder=False, eval_metric="logloss", early_stopping_rounds=50)
    model.fit(x_train, y_train, eval_set=[(x_val, y_val)], verbose=False)
    y_pred = model.predict_proba(x_val)[:, 1]
    return brier_score_loss(y_val, y_pred)

# Split men's training data for hyperparameter tuning
x_train_men_tr, x_val_men, y_train_men_tr, y_val_men = train_test_split(
    x_train_men, y_train_men, test_size=0.2, random_state=42
)

# Run optimization for men
study_men = optuna.create_study(direction="minimize")
study_men.optimize(lambda trial: objective_xgb_men(trial, x_train_men_tr, y_train_men_tr, x_val_men, y_val_men), n_trials=50)
best_params_men_xgb = study_men.best_params
print("Best XGBoost Params (Men):", best_params_men_xgb)

# Train final XGBoost model for men with best hyperparameters
best_xgb_men = XGBClassifier(**best_params_men_xgb, use_label_encoder=False, eval_metric="logloss")
best_xgb_men.fit(x_train_men, y_train_men)
y_pred_men_xgb = best_xgb_men.predict_proba(x_test_men)[:, 1]
brier_men_xgb = brier_score_loss(y_test_men, y_pred_men_xgb)
print("Best XGBoost Brier Score (Men):", brier_men_xgb)


[I 2025-03-19 20:55:10,434] A new study created in memory with name: no-name-7d67e4f3-0f04-41c8-b583-857ef729369d
[I 2025-03-19 20:55:11,120] Trial 0 finished with value: 0.1385477582930327 and parameters: {'n_estimators': 452, 'max_depth': 8, 'learning_rate': 0.041149576347191075, 'subsample': 0.7659335031475915, 'colsample_bytree': 0.8554189437716251, 'min_child_weight': 4}. Best is trial 0 with value: 0.1385477582930327.
[I 2025-03-19 20:55:11,418] Trial 1 finished with value: 0.13731535171502335 and parameters: {'n_estimators': 388, 'max_depth': 3, 'learning_rate': 0.09791080252808289, 'subsample': 0.9647678464196416, 'colsample_bytree': 0.868027842625448, 'min_child_weight': 4}. Best is trial 1 with value: 0.13731535171502335.
[I 2025-03-19 20:55:12,606] Trial 2 finished with value: 0.13483713726084257 and parameters: {'n_estimators': 494, 'max_depth': 6, 'learning_rate': 0.011167340912621794, 'subsample': 0.8202462666419732, 'colsample_bytree': 0.6591140421897838, 'min_child_weig

[I 2025-03-19 20:55:28,494] Trial 26 finished with value: 0.13309415286491663 and parameters: {'n_estimators': 540, 'max_depth': 3, 'learning_rate': 0.014841957914137172, 'subsample': 0.785121915721602, 'colsample_bytree': 0.901379587105358, 'min_child_weight': 10}. Best is trial 21 with value: 0.13127611048952853.
[I 2025-03-19 20:55:29,213] Trial 27 finished with value: 0.1334142897977518 and parameters: {'n_estimators': 674, 'max_depth': 4, 'learning_rate': 0.03480898865171312, 'subsample': 0.6736869059097167, 'colsample_bytree': 0.7071821114564066, 'min_child_weight': 18}. Best is trial 21 with value: 0.13127611048952853.
[I 2025-03-19 20:55:29,977] Trial 28 finished with value: 0.13331711559674522 and parameters: {'n_estimators': 429, 'max_depth': 3, 'learning_rate': 0.01648389366882357, 'subsample': 0.7389169517892609, 'colsample_bytree': 0.6494766793719973, 'min_child_weight': 13}. Best is trial 21 with value: 0.13127611048952853.
[I 2025-03-19 20:55:30,444] Trial 29 finished wi

Best XGBoost Params (Women): {'n_estimators': 640, 'max_depth': 3, 'learning_rate': 0.03005203686760732, 'subsample': 0.6866120720239999, 'colsample_bytree': 0.7928415430518504, 'min_child_weight': 13}


[I 2025-03-19 20:55:44,908] A new study created in memory with name: no-name-1b9d9e3b-264a-4fb5-bcef-80c437bfd66f


Best XGBoost Brier Score (Women): 0.1756633538553978


[I 2025-03-19 20:55:45,445] Trial 0 finished with value: 0.1802026744840222 and parameters: {'n_estimators': 366, 'max_depth': 8, 'learning_rate': 0.09306344650176585, 'subsample': 0.9454026898564322, 'colsample_bytree': 0.8069236012375829, 'min_child_weight': 10}. Best is trial 0 with value: 0.1802026744840222.
[I 2025-03-19 20:55:46,106] Trial 1 finished with value: 0.18198969128763098 and parameters: {'n_estimators': 690, 'max_depth': 9, 'learning_rate': 0.059362390800145666, 'subsample': 0.9727770305565143, 'colsample_bytree': 0.7268145711834807, 'min_child_weight': 14}. Best is trial 0 with value: 0.1802026744840222.
[I 2025-03-19 20:55:47,022] Trial 2 finished with value: 0.1815228513392422 and parameters: {'n_estimators': 880, 'max_depth': 7, 'learning_rate': 0.030355173704132536, 'subsample': 0.9148957242684468, 'colsample_bytree': 0.8110548092056176, 'min_child_weight': 13}. Best is trial 0 with value: 0.1802026744840222.
[I 2025-03-19 20:55:47,704] Trial 3 finished with value

[I 2025-03-19 20:56:03,731] Trial 26 finished with value: 0.17892277236549875 and parameters: {'n_estimators': 327, 'max_depth': 4, 'learning_rate': 0.011595655320496655, 'subsample': 0.7018863719389511, 'colsample_bytree': 0.8374567006297905, 'min_child_weight': 15}. Best is trial 18 with value: 0.17573368167848938.
[I 2025-03-19 20:56:04,506] Trial 27 finished with value: 0.17934337801197886 and parameters: {'n_estimators': 182, 'max_depth': 8, 'learning_rate': 0.02624438179691359, 'subsample': 0.8386228117674179, 'colsample_bytree': 0.7743433715375454, 'min_child_weight': 18}. Best is trial 18 with value: 0.17573368167848938.
[I 2025-03-19 20:56:04,927] Trial 28 finished with value: 0.18147225438783765 and parameters: {'n_estimators': 105, 'max_depth': 5, 'learning_rate': 0.02094789811795031, 'subsample': 0.6360687597838487, 'colsample_bytree': 0.9553609160927097, 'min_child_weight': 19}. Best is trial 18 with value: 0.17573368167848938.
[I 2025-03-19 20:56:05,938] Trial 29 finished

Best XGBoost Params (Men): {'n_estimators': 281, 'max_depth': 6, 'learning_rate': 0.03592649615636062, 'subsample': 0.6497088651320888, 'colsample_bytree': 0.8658988386604175, 'min_child_weight': 20}
Best XGBoost Brier Score (Men): 0.21571874886930334


## Gradient boosting

In [40]:
import optuna
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import brier_score_loss
from sklearn.model_selection import train_test_split

def objective_gbm(trial, x_train, y_train, x_val, y_val):
    params = {
        "n_estimators": trial.suggest_int("n_estimators", 100, 1000),
        "max_depth": trial.suggest_int("max_depth", 3, 10),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.1, log=True),  # narrow range to slow learning
        "subsample": trial.suggest_float("subsample", 0.6, 1.0),
        "max_features": trial.suggest_float("max_features", 0.6, 1.0),
        "min_samples_split": trial.suggest_int("min_samples_split", 2, 20)
    }
    
    model = GradientBoostingClassifier(**params, random_state=42)
    model.fit(x_train, y_train)
    y_pred = model.predict_proba(x_val)[:, 1]
    return brier_score_loss(y_val, y_pred)

# Split women's training data for hyperparameter tuning
x_train_women_tr, x_val_women, y_train_women_tr, y_val_women = train_test_split(
    x_train_women, y_train_women, test_size=0.2, random_state=42
)

# Run optimization for women
study_women_gbm = optuna.create_study(direction="minimize")
study_women_gbm.optimize(lambda trial: objective_gbm(trial, x_train_women_tr, y_train_women_tr, x_val_women, y_val_women), n_trials=50)
best_params_women_gbm = study_women_gbm.best_params
print("Best Gradient Boosting Params (Women):", best_params_women_gbm)

# Train final model with best hyperparameters for women
best_gbm_women = GradientBoostingClassifier(**best_params_women_gbm, random_state=42)
best_gbm_women.fit(x_train_women, y_train_women)
y_pred_women_gbm = best_gbm_women.predict_proba(x_test_women)[:, 1]
brier_women_gbm = brier_score_loss(y_test_women, y_pred_women_gbm)
print("Gradient Boosting Brier Score (Women):", brier_women_gbm)


[I 2025-03-19 21:30:00,948] A new study created in memory with name: no-name-310feef2-e40c-46a7-bef8-cc245372edeb
[I 2025-03-19 21:30:15,654] Trial 0 finished with value: 0.17692169033961463 and parameters: {'n_estimators': 777, 'max_depth': 7, 'learning_rate': 0.0327567029529404, 'subsample': 0.8417960432938064, 'max_features': 0.8303127925364528, 'min_samples_split': 4}. Best is trial 0 with value: 0.17692169033961463.
[I 2025-03-19 21:30:36,433] Trial 1 finished with value: 0.20112883893474418 and parameters: {'n_estimators': 923, 'max_depth': 10, 'learning_rate': 0.04091861082450788, 'subsample': 0.8049989273076641, 'max_features': 0.7566344537733701, 'min_samples_split': 12}. Best is trial 0 with value: 0.17692169033961463.
[I 2025-03-19 21:30:41,271] Trial 2 finished with value: 0.14195290041324435 and parameters: {'n_estimators': 240, 'max_depth': 8, 'learning_rate': 0.010960583977933971, 'subsample': 0.9381046406872368, 'max_features': 0.6940286971025742, 'min_samples_split': 8

[I 2025-03-19 21:32:24,644] Trial 26 finished with value: 0.15339306334021846 and parameters: {'n_estimators': 302, 'max_depth': 6, 'learning_rate': 0.029367219741845023, 'subsample': 0.6357898978371634, 'max_features': 0.9140250867106945, 'min_samples_split': 2}. Best is trial 18 with value: 0.13522970141565974.
[I 2025-03-19 21:32:31,884] Trial 27 finished with value: 0.19082787664092743 and parameters: {'n_estimators': 627, 'max_depth': 5, 'learning_rate': 0.0983706664800132, 'subsample': 0.6197765454150188, 'max_features': 0.981077345945377, 'min_samples_split': 18}. Best is trial 18 with value: 0.13522970141565974.
[I 2025-03-19 21:32:36,410] Trial 28 finished with value: 0.14467404680264892 and parameters: {'n_estimators': 454, 'max_depth': 4, 'learning_rate': 0.037362838147411766, 'subsample': 0.6962103581355472, 'max_features': 0.9531527914285851, 'min_samples_split': 11}. Best is trial 18 with value: 0.13522970141565974.
[I 2025-03-19 21:32:40,613] Trial 29 finished with value

Best Gradient Boosting Params (Women): {'n_estimators': 224, 'max_depth': 4, 'learning_rate': 0.03439690121626746, 'subsample': 0.6022584731915432, 'max_features': 0.9363111160870559, 'min_samples_split': 16}
Gradient Boosting Brier Score (Women): 0.1816350908384973


In [41]:
def objective_gbm_men(trial, x_train, y_train, x_val, y_val):
    # We can use the same hyperparameter search space as for women.
    params = {
        "n_estimators": trial.suggest_int("n_estimators", 100, 1000),
        "max_depth": trial.suggest_int("max_depth", 3, 10),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.1, log=True),
        "subsample": trial.suggest_float("subsample", 0.6, 1.0),
        "max_features": trial.suggest_float("max_features", 0.6, 1.0),
        "min_samples_split": trial.suggest_int("min_samples_split", 2, 20)
    }
    
    model = GradientBoostingClassifier(**params, random_state=42)
    model.fit(x_train, y_train)
    y_pred = model.predict_proba(x_val)[:, 1]
    return brier_score_loss(y_val, y_pred)

# Split men's training data for hyperparameter tuning
x_train_men_tr, x_val_men, y_train_men_tr, y_val_men = train_test_split(
    x_train_men, y_train_men, test_size=0.2, random_state=42
)

# Run optimization for men
study_men_gbm = optuna.create_study(direction="minimize")
study_men_gbm.optimize(lambda trial: objective_gbm_men(trial, x_train_men_tr, y_train_men_tr, x_val_men, y_val_men), n_trials=50)
best_params_men_gbm = study_men_gbm.best_params
print("Best Gradient Boosting Params (Men):", best_params_men_gbm)

# Train final model with best hyperparameters for men
best_gbm_men = GradientBoostingClassifier(**best_params_men_gbm, random_state=42)
best_gbm_men.fit(x_train_men, y_train_men)
y_pred_men_gbm = best_gbm_men.predict_proba(x_test_men)[:, 1]
brier_men_gbm = brier_score_loss(y_test_men, y_pred_men_gbm)
print("Gradient Boosting Brier Score (Men):", brier_men_gbm)


[I 2025-03-19 21:34:01,794] A new study created in memory with name: no-name-b130497c-495c-4e65-aeeb-8620b8959258
[I 2025-03-19 21:34:18,287] Trial 0 finished with value: 0.22019154577498318 and parameters: {'n_estimators': 602, 'max_depth': 8, 'learning_rate': 0.04726249402270229, 'subsample': 0.7627556999373163, 'max_features': 0.7379639500820088, 'min_samples_split': 20}. Best is trial 0 with value: 0.22019154577498318.
[I 2025-03-19 21:34:24,471] Trial 1 finished with value: 0.18071187965322058 and parameters: {'n_estimators': 523, 'max_depth': 3, 'learning_rate': 0.014807483475452021, 'subsample': 0.7075225989679703, 'max_features': 0.8786614946843745, 'min_samples_split': 4}. Best is trial 1 with value: 0.18071187965322058.
[I 2025-03-19 21:35:08,654] Trial 2 finished with value: 0.23766918115659766 and parameters: {'n_estimators': 935, 'max_depth': 10, 'learning_rate': 0.028744285690933985, 'subsample': 0.897685722884354, 'max_features': 0.8775344798432816, 'min_samples_split': 

[I 2025-03-19 21:39:00,441] Trial 26 finished with value: 0.18427923118753825 and parameters: {'n_estimators': 234, 'max_depth': 5, 'learning_rate': 0.010229159555592338, 'subsample': 0.7806086661463131, 'max_features': 0.8768249891538318, 'min_samples_split': 4}. Best is trial 23 with value: 0.1798622132076808.
[I 2025-03-19 21:39:02,760] Trial 27 finished with value: 0.18368062169850233 and parameters: {'n_estimators': 133, 'max_depth': 4, 'learning_rate': 0.01564947115715603, 'subsample': 0.7490318831208804, 'max_features': 0.8519610334951365, 'min_samples_split': 9}. Best is trial 23 with value: 0.1798622132076808.
[I 2025-03-19 21:39:07,299] Trial 28 finished with value: 0.18243805089730858 and parameters: {'n_estimators': 225, 'max_depth': 6, 'learning_rate': 0.012945997807544098, 'subsample': 0.6382507954705141, 'max_features': 0.7718534682774302, 'min_samples_split': 6}. Best is trial 23 with value: 0.1798622132076808.
[I 2025-03-19 21:39:11,246] Trial 29 finished with value: 0

Best Gradient Boosting Params (Men): {'n_estimators': 276, 'max_depth': 3, 'learning_rate': 0.020493663774026, 'subsample': 0.8239412108576687, 'max_features': 0.6092262903420373, 'min_samples_split': 4}
Gradient Boosting Brier Score (Men): 0.2113394054272737


## Logistic regression

In [8]:
def objective_lr(trial, x_train, y_train, x_val, y_val):
    C = trial.suggest_float("C", 1e-4, 10, log=True)
    
    # Define the model pipeline with scaling
    model = Pipeline([
        ("scaler", StandardScaler()),
        ("classifier", LogisticRegression(C=C, solver="liblinear", max_iter=1000))
    ])
    
    # Train the model
    model.fit(x_train, y_train)
    
    # Predict probabilities
    y_pred = model.predict_proba(x_val)[:, 1]
    
    # Return Brier Score as the optimization metric
    return brier_score_loss(y_val, y_pred)


###################################################### WOMEN

# Split data for tuning
x_train_women_tr, x_val_women, y_train_women_tr, y_val_women = train_test_split(
    x_train_women, y_train_women, test_size=0.2, random_state=42
)

# Run Optuna optimization
study = optuna.create_study(direction="minimize")
study.optimize(lambda trial: objective_lr(trial, x_train_women_tr, y_train_women_tr, x_val_women, y_val_women), n_trials=50)

# Get best parameters
best_params_women_lr = study.best_params
print("Best Logistic Regression Params (Women):", best_params_women_lr)

# Train final Logistic Regression model with best params
best_lr_women = Pipeline([
    ("scaler", StandardScaler()),
    ("classifier", LogisticRegression(C=best_params_women_lr["C"], solver="liblinear", max_iter=1000))
])
best_lr_women.fit(x_train_women, y_train_women)

# Predict on test set
y_pred_women_lr = best_lr_women.predict_proba(x_test_women)[:, 1]

# Compute Brier Score
brier_women_lr = brier_score_loss(y_test_women, y_pred_women_lr)
print("Best Logistic Regression Brier Score (Women):", brier_women_lr)


###################################################### MEN

# Split data for tuning
x_train_men_tr, x_val_men, y_train_men_tr, y_val_men = train_test_split(
    x_train_men, y_train_men, test_size=0.2, random_state=42
)

# Run Optuna optimization
study = optuna.create_study(direction="minimize")
study.optimize(lambda trial: objective_lr(trial, x_train_men_tr, y_train_men_tr, x_val_men, y_val_men), n_trials=50)

# Get best parameters
best_params_men_lr = study.best_params
print("Best Logistic Regression Params (Men):", best_params_men_lr)

# Train final Logistic Regression model with best params
best_lr_men = Pipeline([
    ("scaler", StandardScaler()),
    ("classifier", LogisticRegression(C=best_params_men_lr["C"], solver="liblinear", max_iter=1000))
])
best_lr_men.fit(x_train_men, y_train_men)

# Predict on test set
y_pred_men_lr = best_lr_men.predict_proba(x_test_men)[:, 1]

# Compute Brier Score
brier_men_lr = brier_score_loss(y_test_men, y_pred_men_lr)
print("Best Logistic Regression Brier Score (Men):", brier_men_lr)

[I 2025-03-19 20:56:19,163] A new study created in memory with name: no-name-ff7732af-f13e-4108-9a43-2e908350861a
[I 2025-03-19 20:56:19,186] Trial 0 finished with value: 0.13895981434986568 and parameters: {'C': 0.17561453182839584}. Best is trial 0 with value: 0.13895981434986568.
[I 2025-03-19 20:56:19,205] Trial 1 finished with value: 0.13825498400739936 and parameters: {'C': 0.02989259244643371}. Best is trial 1 with value: 0.13825498400739936.
[I 2025-03-19 20:56:19,209] Trial 2 finished with value: 0.1387036100875501 and parameters: {'C': 0.01889535036924333}. Best is trial 1 with value: 0.13825498400739936.
[I 2025-03-19 20:56:19,236] Trial 3 finished with value: 0.1417345449356082 and parameters: {'C': 0.007656293168645518}. Best is trial 1 with value: 0.13825498400739936.
[I 2025-03-19 20:56:19,255] Trial 4 finished with value: 0.13953872485359517 and parameters: {'C': 0.3572933084261202}. Best is trial 1 with value: 0.13825498400739936.
[I 2025-03-19 20:56:19,276] Trial 5 fi

[I 2025-03-19 20:56:20,110] Trial 48 finished with value: 0.14033263645880262 and parameters: {'C': 0.010258665922030346}. Best is trial 14 with value: 0.1381852295215624.
[I 2025-03-19 20:56:20,131] Trial 49 finished with value: 0.14017416948214662 and parameters: {'C': 0.7816548374359485}. Best is trial 14 with value: 0.1381852295215624.
[I 2025-03-19 20:56:20,158] A new study created in memory with name: no-name-51e82eb8-551b-4d83-94ec-a3c57570733b
[I 2025-03-19 20:56:20,182] Trial 0 finished with value: 0.1774575381094675 and parameters: {'C': 0.021393265844594223}. Best is trial 0 with value: 0.1774575381094675.
[I 2025-03-19 20:56:20,222] Trial 1 finished with value: 0.17925565185533776 and parameters: {'C': 3.704831554713742}. Best is trial 0 with value: 0.1774575381094675.
[I 2025-03-19 20:56:20,232] Trial 2 finished with value: 0.22911852624445248 and parameters: {'C': 0.00012903813275553117}. Best is trial 0 with value: 0.1774575381094675.
[I 2025-03-19 20:56:20,266] Trial 3 

Best Logistic Regression Params (Women): {'C': 0.042491537234950194}
Best Logistic Regression Brier Score (Women): 0.160885434259162


[I 2025-03-19 20:56:20,348] Trial 8 finished with value: 0.17924120528630258 and parameters: {'C': 3.212226702119603}. Best is trial 0 with value: 0.1774575381094675.
[I 2025-03-19 20:56:20,380] Trial 9 finished with value: 0.17890762545175937 and parameters: {'C': 0.705098369586604}. Best is trial 0 with value: 0.1774575381094675.
[I 2025-03-19 20:56:20,397] Trial 10 finished with value: 0.17750863730776292 and parameters: {'C': 0.06122486678260656}. Best is trial 0 with value: 0.1774575381094675.
[I 2025-03-19 20:56:20,430] Trial 11 finished with value: 0.17747829228691292 and parameters: {'C': 0.05713134435184145}. Best is trial 0 with value: 0.1774575381094675.
[I 2025-03-19 20:56:20,448] Trial 12 finished with value: 0.17874373047018785 and parameters: {'C': 0.008324158478903847}. Best is trial 0 with value: 0.1774575381094675.
[I 2025-03-19 20:56:20,478] Trial 13 finished with value: 0.17803183107974804 and parameters: {'C': 0.14596359507919676}. Best is trial 0 with value: 0.177

Best Logistic Regression Params (Men): {'C': 0.035308922366192014}
Best Logistic Regression Brier Score (Men): 0.20424594469266563


## Neural Net

In [9]:
import optuna
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import brier_score_loss
from sklearn.model_selection import train_test_split
import numpy as np

# ------------------------------ FOR WOMEN ------------------------------

def objective_nn(trial, x_train, y_train, x_val, y_val):
    # Define hyperparameters for the neural network.
    # Hidden layers: We choose the number of layers and neurons per layer.
    n_layers = trial.suggest_int("n_layers", 1, 3)
    hidden_layer_sizes = []
    for i in range(n_layers):
        neurons = trial.suggest_int(f"n_units_l{i}", 10, 200)
        hidden_layer_sizes.append(neurons)
    hidden_layer_sizes = tuple(hidden_layer_sizes)
    
    # Regularization strength
    alpha = trial.suggest_float("alpha", 1e-2, 1.0, log=True)
    
    # Initial learning rate
    learning_rate_init = trial.suggest_float("learning_rate_init", 1e-4, 1e-1, log=True)
    
    # Activation function: 'relu' or 'tanh' are common choices.
    activation = trial.suggest_categorical("activation", ["relu", "tanh"])
    
    # Build a pipeline with StandardScaler and MLPClassifier
    model = Pipeline([
        ("scaler", StandardScaler()),
        ("mlp", MLPClassifier(
            hidden_layer_sizes=hidden_layer_sizes,
            activation=activation,
            alpha=alpha,
            learning_rate_init=learning_rate_init,
            max_iter=1000,
            random_state=42
        ))
    ])
    
    # Fit the model on the training split
    model.fit(x_train, y_train)
    
    # Predict probabilities on the validation split
    y_pred = model.predict_proba(x_val)[:, 1]
    
    # Return Brier Score (lower is better)
    return brier_score_loss(y_val, y_pred)

# Split women's training data into train and validation sets
x_train_women_tr, x_val_women, y_train_women_tr, y_val_women = train_test_split(
    x_train_women, y_train_women, test_size=0.2, random_state=42
)

# Optimize neural network hyperparameters for women
study_women_nn = optuna.create_study(direction="minimize")
study_women_nn.optimize(lambda trial: objective_nn(trial, x_train_women_tr, y_train_women_tr, x_val_women, y_val_women),
                          n_trials=50)
best_params_women_nn = study_women_nn.best_params
print("Best NN Params (Women):", best_params_women_nn)

# Train final NN model for women with best hyperparameters
best_nn_women = Pipeline([
    ("scaler", StandardScaler()),
    ("mlp", MLPClassifier(
        hidden_layer_sizes=tuple(best_params_women_nn[f"n_units_l{i}"] for i in range(best_params_women_nn["n_layers"])),
        activation=best_params_women_nn["activation"],
        alpha=best_params_women_nn["alpha"],
        learning_rate_init=best_params_women_nn["learning_rate_init"],
        max_iter=1000,
        random_state=42
    ))
])
best_nn_women.fit(x_train_women, y_train_women)
y_pred_women_nn = best_nn_women.predict_proba(x_test_women)[:, 1]
brier_women_nn = brier_score_loss(y_test_women, y_pred_women_nn)
print("Best NN Brier Score (Women):", brier_women_nn)

# ------------------------------ FOR MEN ------------------------------

def objective_nn_men(trial, x_train, y_train, x_val, y_val):
    # Use the same search space as for women
    n_layers = trial.suggest_int("n_layers", 1, 3)
    hidden_layer_sizes = []
    for i in range(n_layers):
        neurons = trial.suggest_int(f"n_units_l{i}", 10, 100)
        hidden_layer_sizes.append(neurons)
    hidden_layer_sizes = tuple(hidden_layer_sizes)
    
    alpha = trial.suggest_float("alpha", 1e-5, 1e-1, log=True)
    learning_rate_init = trial.suggest_float("learning_rate_init", 1e-4, 1e-1, log=True)
    activation = trial.suggest_categorical("activation", ["relu", "tanh"])
    
    model = Pipeline([
        ("scaler", StandardScaler()),
        ("mlp", MLPClassifier(
            hidden_layer_sizes=hidden_layer_sizes,
            activation=activation,
            alpha=alpha,
            learning_rate_init=learning_rate_init,
            max_iter=1000,
            random_state=42
        ))
    ])
    model.fit(x_train, y_train)
    y_pred = model.predict_proba(x_val)[:, 1]
    return brier_score_loss(y_val, y_pred)

# Split men's training data for tuning
x_train_men_tr, x_val_men, y_train_men_tr, y_val_men = train_test_split(
    x_train_men, y_train_men, test_size=0.2, random_state=42
)

study_men_nn = optuna.create_study(direction="minimize")
study_men_nn.optimize(lambda trial: objective_nn_men(trial, x_train_men_tr, y_train_men_tr, x_val_men, y_val_men),
                        n_trials=50)
best_params_men_nn = study_men_nn.best_params
print("Best NN Params (Men):", best_params_men_nn)

# Train final NN model for men with best hyperparameters
best_nn_men = Pipeline([
    ("scaler", StandardScaler()),
    ("mlp", MLPClassifier(
        hidden_layer_sizes=tuple(best_params_men_nn[f"n_units_l{i}"] for i in range(best_params_men_nn["n_layers"])),
        activation=best_params_men_nn["activation"],
        alpha=best_params_men_nn["alpha"],
        learning_rate_init=best_params_men_nn["learning_rate_init"],
        max_iter=1000,
        random_state=42
    ))
])
best_nn_men.fit(x_train_men, y_train_men)
y_pred_men_nn = best_nn_men.predict_proba(x_test_men)[:, 1]
brier_men_nn = brier_score_loss(y_test_men, y_pred_men_nn)
print("Best NN Brier Score (Men):", brier_men_nn)


[I 2025-03-19 20:56:21,381] A new study created in memory with name: no-name-907358e1-70c0-4d56-8a3d-7df434d38900
[I 2025-03-19 20:56:47,892] Trial 0 finished with value: 0.22106958900965995 and parameters: {'n_layers': 2, 'n_units_l0': 145, 'n_units_l1': 138, 'alpha': 0.0358938435629133, 'learning_rate_init': 0.00015395491532400706, 'activation': 'tanh'}. Best is trial 0 with value: 0.22106958900965995.
[I 2025-03-19 20:56:52,199] Trial 1 finished with value: 0.13885112161802735 and parameters: {'n_layers': 1, 'n_units_l0': 176, 'alpha': 0.5237074082933967, 'learning_rate_init': 0.00042720595880633, 'activation': 'tanh'}. Best is trial 1 with value: 0.13885112161802735.
[I 2025-03-19 20:56:55,734] Trial 2 finished with value: 0.19571428422311568 and parameters: {'n_layers': 2, 'n_units_l0': 152, 'n_units_l1': 52, 'alpha': 0.48768658012403554, 'learning_rate_init': 0.0008647238448875027, 'activation': 'relu'}. Best is trial 1 with value: 0.13885112161802735.
[I 2025-03-19 20:57:03,962]

[I 2025-03-19 20:58:36,876] Trial 27 finished with value: 0.13915012988181066 and parameters: {'n_layers': 1, 'n_units_l0': 30, 'alpha': 0.7754766104924535, 'learning_rate_init': 0.00010472816153789773, 'activation': 'tanh'}. Best is trial 22 with value: 0.13768048813364323.
C:\Users\Sebastian\anaconda3\lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:692: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (1000) reached and the optimization hasn't converged yet.
  warnings.warn(
[I 2025-03-19 20:58:52,252] Trial 28 finished with value: 0.2127486456833359 and parameters: {'n_layers': 2, 'n_units_l0': 50, 'n_units_l1': 66, 'alpha': 0.22753901534154294, 'learning_rate_init': 0.0002578024787497104, 'activation': 'tanh'}. Best is trial 22 with value: 0.13768048813364323.
[I 2025-03-19 20:58:52,354] Trial 29 finished with value: 0.15473379204175958 and parameters: {'n_layers': 1, 'n_units_l0': 22, 'alpha': 0.40177595081385664, 'learning_rate_init': 0.099870686341

[I 2025-03-19 21:01:18,133] Trial 48 finished with value: 0.13929546830661862 and parameters: {'n_layers': 1, 'n_units_l0': 64, 'alpha': 0.9866742809223547, 'learning_rate_init': 0.0002186970762548784, 'activation': 'tanh'}. Best is trial 46 with value: 0.1376288694231072.
[I 2025-03-19 21:01:24,226] Trial 49 finished with value: 0.19215909578392754 and parameters: {'n_layers': 3, 'n_units_l0': 99, 'n_units_l1': 76, 'n_units_l2': 25, 'alpha': 0.3696849641523984, 'learning_rate_init': 0.00042530459322589456, 'activation': 'tanh'}. Best is trial 46 with value: 0.1376288694231072.


Best NN Params (Women): {'n_layers': 1, 'n_units_l0': 64, 'alpha': 0.8003873186888315, 'learning_rate_init': 0.00035193893542366804, 'activation': 'tanh'}


C:\Users\Sebastian\anaconda3\lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:692: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (1000) reached and the optimization hasn't converged yet.
  warnings.warn(
[I 2025-03-19 21:01:34,788] A new study created in memory with name: no-name-e8a972eb-a256-4239-9aee-10d876f0eccf


...

[I 2025-03-19 21:01:36,131] Trial 0 finished with value: 0.2530192558564613 and parameters: {'n_layers': 2, 'n_units_l0': 85, 'n_units_l1': 30, 'alpha': 0.0007742813261237722, 'learning_rate_init': 0.02669562802650229, 'activation': 'relu'}. Best is trial 0 with value: 0.2530192558564613.
[I 2025-03-19 21:01:38,323] Trial 1 finished with value: 0.2751741417569776 and parameters: {'n_layers': 2, 'n_units_l0': 94, 'n_units_l1': 40, 'alpha': 4.305969368210706e-05, 'learning_rate_init': 0.006097824862262933, 'activation': 'relu'}. Best is trial 0 with value: 0.2530192558564613.
[I 2025-03-19 21:01:39,600] Trial 2 finished with value: 0.22833810298128396 and parameters: {'n_layers': 2, 'n_units_l0': 75, 'n_units_l1': 98, 'alpha': 0.0031359836103956994, 'learning_rate_init': 0.04016201746926704, 'activation': 'relu'}. Best is trial 2 with value: 0.22833810298128396.
[I 2025-03-19 21:01:43,587] Trial 3 finished with value: 0.324395756923383 and parameters: {'n_layers': 2, 'n_units_l0': 30, 'n

C:\Users\Sebastian\anaconda3\lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:692: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (1000) reached and the optimization hasn't converged yet.
  warnings.warn(
[I 2025-03-19 21:05:38,390] Trial 21 finished with value: 0.19459869148913636 and parameters: {'n_layers': 3, 'n_units_l0': 21, 'n_units_l1': 57, 'n_units_l2': 26, 'alpha': 1.6050092410354668e-05, 'learning_rate_init': 0.00010592822608500664, 'activation': 'tanh'}. Best is trial 21 with value: 0.19459869148913636.
[I 2025-03-19 21:05:38,908] Trial 22 finished with value: 0.20370733901553878 and parameters: {'n_layers': 3, 'n_units_l0': 11, 'n_units_l1': 52, 'n_units_l2': 11, 'alpha': 2.938509191205329e-05, 'learning_rate_init': 0.09886495143078318, 'activation': 'tanh'}. Best is trial 21 with value: 0.19459869148913636.
[I 2025-03-19 21:05:48,736] Trial 23 finished with value: 0.35598293670224185 and parameters: {'n_layers': 3, 'n_units_l0': 27, 'n_unit

C:\Users\Sebastian\anaconda3\lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:692: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (1000) reached and the optimization hasn't converged yet.
  warnings.warn(
[I 2025-03-19 21:07:39,154] Trial 47 finished with value: 0.2940536470759242 and parameters: {'n_layers': 3, 'n_units_l0': 15, 'n_units_l1': 43, 'n_units_l2': 18, 'alpha': 0.0004984017428910337, 'learning_rate_init': 0.00035730993914343933, 'activation': 'tanh'}. Best is trial 21 with value: 0.19459869148913636.
C:\Users\Sebastian\anaconda3\lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:692: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (1000) reached and the optimization hasn't converged yet.
  warnings.warn(
[I 2025-03-19 21:08:16,426] Trial 48 finished with value: 0.3080364789744523 and parameters: {'n_layers': 3, 'n_units_l0': 36, 'n_units_l1': 66, 'n_units_l2': 81, 'alpha': 1.523949499213055e-05, 'learning_rate_in

Best NN Params (Men): {'n_layers': 3, 'n_units_l0': 21, 'n_units_l1': 57, 'n_units_l2': 26, 'alpha': 1.6050092410354668e-05, 'learning_rate_init': 0.00010592822608500664, 'activation': 'tanh'}
Best NN Brier Score (Men): 0.2345840883727847


C:\Users\Sebastian\anaconda3\lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:692: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (1000) reached and the optimization hasn't converged yet.
  warnings.warn(


## Ensamble

In [ ]:
import numpy as np
from sklearn.ensemble import StackingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import RandomizedSearchCV
from sklearn.metrics import brier_score_loss

# ------------------- For Women -------------------
# Define base estimators for women. They should already be tuned/trained.
estimators_women = [
#     ('catboost', best_catboost_women),
    ('xgboost', best_xgb_women),
    ('logistic', best_lr_women),
#     ('neural', best_nn_women),
]

# Build stacking classifier using a logistic regression meta-learner.
stacking_women = StackingClassifier(
    estimators=estimators_women,
    final_estimator=LogisticRegression(solver='liblinear', max_iter=1000),
    passthrough=False  # Change to True if you want meta-learner to see original features as well
)

# Define hyperparameter distribution for the meta-learner (final_estimator).
param_distributions_women = {
    'final_estimator__C': np.logspace(-4, 1, 50)
}

# Optimize meta-learner using RandomizedSearchCV
search_women = RandomizedSearchCV(
    estimator=stacking_women,
    param_distributions=param_distributions_women,
    n_iter=5,
    scoring='neg_log_loss',  # optimizing for well-calibrated probabilities
    cv=5,
    random_state=42,
    n_jobs=-1,
    verbose=1
)
search_women.fit(x_train_women, y_train_women)
best_stacking_women = search_women.best_estimator_

# Evaluate on test set
y_pred_women_stacking = best_stacking_women.predict_proba(x_test_women)[:, 1]
brier_women_stacking = brier_score_loss(y_test_women, y_pred_women_stacking)
print("Best Stacking Classifier Brier Score (Women):", brier_women_stacking)


# ------------------- For Men -------------------
# Define base estimators for men. They should already be tuned/trained.
estimators_men = [
#     ('catboost', best_catboost_men),
    ('xgboost', best_xgb_men),
    ('logistic', best_lr_men),
#     ('neural', best_nn_men),
]

# Build stacking classifier using a logistic regression meta-learner.
stacking_men = StackingClassifier(
    estimators=estimators_men,
    final_estimator=LogisticRegression(solver='liblinear', max_iter=1000),
    passthrough=False
)

# Define hyperparameter distribution for the meta-learner.
param_distributions_men = {
    'final_estimator__C': np.logspace(-4, 1, 50)
}

# Optimize using RandomizedSearchCV
search_men = RandomizedSearchCV(
    estimator=stacking_men,
    param_distributions=param_distributions_men,
    n_iter=5,
    scoring='neg_log_loss',
    cv=5,
    random_state=42,
    n_jobs=-1,
    verbose=1
)
search_men.fit(x_train_men, y_train_men)
best_stacking_men = search_men.best_estimator_

# Evaluate on test set
y_pred_men_stacking = best_stacking_men.predict_proba(x_test_men)[:, 1]
brier_men_stacking = brier_score_loss(y_test_men, y_pred_men_stacking)
print("Best Stacking Classifier Brier Score (Men):", brier_men_stacking)

In [29]:
print("Final brier:", np.mean((np.concatenate((y_pred_women_stacking, y_pred_men_stacking)) - np.concatenate((y_test_women, y_test_men)))**2))

Final brier: 0.1874275077506849


## Use only models trained on women

In [37]:
import numpy as np
from sklearn.ensemble import StackingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import RandomizedSearchCV
from sklearn.metrics import brier_score_loss

# ------------------- For Women -------------------
# Define base estimators for women. They should already be tuned/trained.
estimators_women = [
#     ('catboost', best_catboost_women),
    ('xgboost', best_xgb_women),
    ('logistic', best_lr_women),
#     ('neural', best_nn_women),
]

# Build stacking classifier using a logistic regression meta-learner.
stacking_women = StackingClassifier(
    estimators=estimators_women,
    final_estimator=LogisticRegression(solver='liblinear', max_iter=1000),
    passthrough=False  # Change to True if you want meta-learner to see original features as well
)

# Define hyperparameter distribution for the meta-learner (final_estimator).
param_distributions_women = {
    'final_estimator__C': np.logspace(-4, 1, 50)
}

# Optimize meta-learner using RandomizedSearchCV
search_women = RandomizedSearchCV(
    estimator=stacking_women,
    param_distributions=param_distributions_women,
    n_iter=5,
    scoring='neg_log_loss',  # optimizing for well-calibrated probabilities
    cv=5,
    random_state=42,
    n_jobs=-1,
    verbose=1
)
search_women.fit(x_train_women, y_train_women)
best_stacking_women = search_women.best_estimator_

# Evaluate on test set
y_pred_women_stacking = best_stacking_women.predict_proba(x_test_women)[:, 1]
brier_women_stacking = brier_score_loss(y_test_women, y_pred_women_stacking)
print("Best Stacking Classifier Brier Score (Women):", brier_women_stacking)

# Evaluate on test set
x_test_men, y_test_men = x_y_from_data_frame(df_test_men_list.iloc[:, :-1])
y_pred_men_stacking = best_stacking_women.predict_proba(x_test_men)[:, 1]
brier_men_stacking = brier_score_loss(y_test_men, y_pred_men_stacking)
print("Best Stacking Classifier Brier Score (Men):", brier_men_stacking)

print("Final brier:", np.mean((np.concatenate((y_pred_women_stacking, y_pred_men_stacking)) - np.concatenate((y_test_women, y_test_men)))**2))

Fitting 5 folds for each of 5 candidates, totalling 25 fits
Best Stacking Classifier Brier Score (Women): 0.16982796590527852
Best Stacking Classifier Brier Score (Men): 0.2076193037237158
Final brier: 0.18927133536259041
